In [2]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
import os

from sklearn.model_selection import train_test_split
from keras.layers import TextVectorization 
from keras.layers import Dropout, Dense, Conv1D, MaxPool1D
from keras.losses import SparseCategoricalCrossentropy
from keras.optimizers import AdamW

from keras.models import Model

I0000 00:00:1780077329.332498  308288 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1780077329.840272  308288 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1780077332.026902  308288 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [3]:
data_root_path = "/mnt/e/Deep Learning/data/news20/20_newsgroup"
print(os.listdir(data_root_path))

['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 'sci.space', 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc']


In [4]:
for directory in os.listdir(data_root_path):
    path = os.path.join(data_root_path, directory)
    print(f"Number of files for directory: {directory} is {len(os.listdir(path))}")

Number of files for directory: alt.atheism is 1000
Number of files for directory: comp.graphics is 1000
Number of files for directory: comp.os.ms-windows.misc is 1000
Number of files for directory: comp.sys.ibm.pc.hardware is 1000
Number of files for directory: comp.sys.mac.hardware is 1000
Number of files for directory: comp.windows.x is 1000
Number of files for directory: misc.forsale is 1000
Number of files for directory: rec.autos is 1000
Number of files for directory: rec.motorcycles is 1000
Number of files for directory: rec.sport.baseball is 1000
Number of files for directory: rec.sport.hockey is 1000
Number of files for directory: sci.crypt is 1000
Number of files for directory: sci.electronics is 1000
Number of files for directory: sci.med is 1000
Number of files for directory: sci.space is 1000
Number of files for directory: soc.religion.christian is 997
Number of files for directory: talk.politics.guns is 1000
Number of files for directory: talk.politics.mideast is 1000
Numb

In [5]:
idx = 100
root_path = os.path.join(data_root_path, "comp.graphics")
list_files = os.listdir(root_path)

file_name = list_files[idx]
file_path = os.path.join(root_path, file_name)

with open(file_path, encoding='latin-1') as f:
    file = f.read()
    print(file)

Newsgroups: comp.graphics
Path: cantaloupe.srv.cs.cmu.edu!crabapple.srv.cs.cmu.edu!bb3.andrew.cmu.edu!news.sei.cmu.edu!cis.ohio-state.edu!zaphod.mps.ohio-state.edu!cs.utexas.edu!utnut!nott!cunews!freenet.carleton.ca!Freenet.carleton.ca!ad994
From: ad994@Freenet.carleton.ca (Jason Wiggle)
Subject: PCX
Message-ID: <1993Apr14.220100.17867@freenet.carleton.ca>
Sender: news@freenet.carleton.ca (News Administrator)
Organization: National Capital Freenet, Ottawa, Canada
Date: Wed, 14 Apr 1993 22:01:00 GMT
Lines: 27


Hello
	HELP!!! please
		I am a student of turbo c++ and graphics programming
	and I am having some problems finding algorithms and code
	to teach me how to do some stuff..

	1) Where is there a book or code that will teach me how
	to read and write pcx,dbf,and gif files?

	2) How do I access the extra ram on my paradise video board
	so I can do paging in the higher vga modes ie: 320x200x256
	800x600x256
	3) anybody got a line on a good book to help answer these question?

Thanks 

In [6]:
lines = file.split("\n")[10:]
file = "\n".join(lines)
file

"\nHello\n\tHELP!!! please\n\t\tI am a student of turbo c++ and graphics programming\n\tand I am having some problems finding algorithms and code\n\tto teach me how to do some stuff..\n\n\t1) Where is there a book or code that will teach me how\n\tto read and write pcx,dbf,and gif files?\n\n\t2) How do I access the extra ram on my paradise video board\n\tso I can do paging in the higher vga modes ie: 320x200x256\n\t800x600x256\n\t3) anybody got a line on a good book to help answer these question?\n\nThanks very much !\n\nsend reply's to : Palm@snycanva.bitnet\n\nPeace be\nBlessed be\nStephen Palm\n"

In [7]:
texts = []
labels = []
class_names = []

for i, class_name in enumerate(sorted(os.listdir(data_root_path))):
    class_names.append(class_name)

    class_path = os.path.join(data_root_path, class_name)

    for file in sorted(os.listdir(class_path)):
        file_path = os.path.join(class_path, file)

        with open(file_path, encoding='latin-1') as f:
            text = f.read()
            lines = text.split("\n")[10:]
            text = "\n".join(lines)
            texts.append(text)
            labels.append(i)

In [8]:
print(f"Number of text files: {len(texts)}")
print(f"Number of labels: {len(labels)}")
print(f"Number of classes: {len(class_names)}")

Number of text files: 19997
Number of labels: 19997
Number of classes: 20


In [9]:
train_texts, val_texts, train_labels, val_labels = train_test_split(texts, labels, test_size=0.2, shuffle=True, random_state=42)

In [10]:
print(f"Training data count: {len(train_texts)}")
print(f"Training labels count: {len(train_labels)}")
print(f"Validation data count: {len(val_texts)}")
print(f"Validation labels count: {len(val_labels)}")

Training data count: 15997
Training labels count: 15997
Validation data count: 4000
Validation labels count: 4000


In [11]:
vectorizer = TextVectorization(max_tokens=20000, output_sequence_length=200)

train_loader = tf.data.Dataset.from_tensor_slices(texts).batch(128)

vectorizer.adapt(train_loader)

E0000 00:00:1780077477.639070  308288 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


In [12]:
vectorizer("Hello world1 This is a test.")

<tf.Tensor: shape=(200,), dtype=int64, numpy=
array([1453,    1,   14,    8,    5,  712,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
     

In [13]:
vectorizer.get_vocabulary()[:5]

['', '[UNK]', np.str_('the'), np.str_('to'), np.str_('of')]

In [14]:
wiki_embed_200_path = "/mnt/e/Deep Learning/data/wiki_giga_2024_200_MFT20_vectors_seed_2024_alpha_0.75_eta_0.05_combined.txt"

dict_embedd = {}

with open(wiki_embed_200_path, 'r') as f:
    for line in f:
        word = line.split(" ")[0]
        vec = np.array([float(str) for str in line.split(" ")[1:]], dtype=np.float32)
        dict_embedd[word] = vec

In [15]:
len(vectorizer.get_vocabulary())

20000

In [16]:
word_index = dict(zip(vectorizer.get_vocabulary(), np.arange(len(vectorizer.get_vocabulary()))))
word_index

{'': np.int64(0),
 '[UNK]': np.int64(1),
 np.str_('the'): np.int64(2),
 np.str_('to'): np.int64(3),
 np.str_('of'): np.int64(4),
 np.str_('a'): np.int64(5),
 np.str_('and'): np.int64(6),
 np.str_('in'): np.int64(7),
 np.str_('is'): np.int64(8),
 np.str_('i'): np.int64(9),
 np.str_('that'): np.int64(10),
 np.str_('it'): np.int64(11),
 np.str_('for'): np.int64(12),
 np.str_('you'): np.int64(13),
 np.str_('this'): np.int64(14),
 np.str_('on'): np.int64(15),
 np.str_('be'): np.int64(16),
 np.str_('not'): np.int64(17),
 np.str_('are'): np.int64(18),
 np.str_('have'): np.int64(19),
 np.str_('with'): np.int64(20),
 np.str_('as'): np.int64(21),
 np.str_('or'): np.int64(22),
 np.str_('if'): np.int64(23),
 np.str_('was'): np.int64(24),
 np.str_('but'): np.int64(25),
 np.str_('they'): np.int64(26),
 np.str_('from'): np.int64(27),
 np.str_('by'): np.int64(28),
 np.str_('at'): np.int64(29),
 np.str_('an'): np.int64(30),
 np.str_('can'): np.int64(31),
 np.str_('my'): np.int64(32),
 np.str_('what'): 

In [17]:
dim = dict_embedd["the"].shape[0]
embeding_matrix = np.zeros((len(vectorizer.get_vocabulary()), dim))
print(embeding_matrix.shape)

(20000, 200)


In [18]:
hits = 0
misses = 0

for word, idx in word_index.items():
    vect = dict_embedd.get(word)
    
    if vect is None:
        misses += 1

    if vect is not None:
        embeding_matrix[idx, :] = vect
        hits += 1

print(f"Number of words that have embeding in glove_200d is: {hits}")
print(f"Number of words that don't have embeding in glove_200d is: {misses}")


Number of words that have embeding in glove_200d is: 18484
Number of words that don't have embeding in glove_200d is: 1516


In [19]:
(embeding_matrix[3] == dict_embedd['the']).sum()

np.int64(0)

In [20]:
(embeding_matrix[3] == dict_embedd['to']).sum()

np.int64(200)

In [21]:
embedding_layer = tf.keras.layers.Embedding(input_dim=len(word_index), 
                                        output_dim=list(dict_embedd.values())[0].shape[0],
                                        embeddings_initializer=tf.keras.initializers.constant(embeding_matrix),
                                        trainable=False)

In [22]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

In [ ]:
input = keras.layers.Input(shape=(None, ), dtype=np.int64)
x = embedding_layer(input)

x = Conv1D(128, 5, activation='relu')(x)
x = MaxPool1D(2)(x)

x = Conv1D(128, 5, activation='relu')(x)
x = MaxPool1D(2)(x)

x = Conv1D(128, 5, activation='relu')(x)

x = tf.keras.layers.GlobalAveragePooling1D()(x)

x = Dense(128, activation='relu')(x)
x = Dropout(0.2)(x)

output = Dense(len(class_names), activation='softmax')(x)

model = Model(input, output)

model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, None, 200)      │     4,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, None, 128)      │       128,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, None, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, None, 128)      │        82,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, None, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, None, 128)      │        82,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 20)             │         2,580 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,311,316 (16.45 MB)

 Trainable params: 311,316 (1.19 MB)

 Non-trainable params: 4,000,000 (15.26 MB)

: 

In [ ]:
train_texts = np.array(train_texts).reshape(-1, 1)
train_texts